# 04 — Retrieval Quality Analysis

Analyses retrieval recall, popularity bias in retrieval, and the relationship between
retrieval failures and generation failures.

**What this notebook covers:**
1. Overall retrieval Recall@K per backend
2. Per-decile retrieval recall — popularity bias in retrieval
3. Recall vs Accuracy scatter per decile
4. Retrieval failure → output failure correlation (per decile & dataset)
5. Popularity skew in retrieval errors
6. Error composition (miss vs hit among wrong answers)
7. Retrieval-output correlation by dataset
8. Three popularity-preference conditions across deciles


In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path().resolve().parent.parent))
from notebooks.full_pipe_eval.shared_setup import *
import matplotlib.ticker as mticker

NUM_DECILES = 10

# ── Detect retrieval metadata ─────────────────────────────────────────────────
def _ensure_hit(df: pd.DataFrame) -> pd.DataFrame:
    """Add boolean 'hit' column from retrieved_doc_ids vs wikipedia_id."""
    df = df.copy()
    if 'hit' in df.columns and df['hit'].notna().any():
        return df
    if 'retrieved_doc_ids' not in df.columns:
        df['hit'] = np.nan
        return df
    if 'wikipedia_id_int' not in df.columns:
        df['wikipedia_id_int'] = pd.to_numeric(
            df.get('wikipedia_id', pd.Series(dtype=float)), errors='coerce'
        )
    def _row_hit(r):
        ids = r.get('retrieved_doc_ids')
        wid = r.get('wikipedia_id_int')
        if not isinstance(ids, list) or len(ids) == 0 or pd.isna(wid):
            return np.nan
        try:
            return int(wid) in [int(x) for x in ids]
        except Exception:
            return np.nan
    df['hit'] = df.apply(_row_hit, axis=1)
    return df

results_all_r = _ensure_hit(results_all)

# Backends that actually have retrieval data
RETRIEVAL_BACKENDS = [
    b for b in BACKEND_KEYS
    if b != 'zero_shot' and
       results_all_r[results_all_r['backend'] == b]['hit'].notna().any()
]
print(f"Backends with retrieval metadata: {RETRIEVAL_BACKENDS}")
if not RETRIEVAL_BACKENDS:
    print("\nWARNING: No retrieval metadata found.  Check that llm_eval_runner "
          "stored 'retrieved_doc_ids' and 'wikipedia_id' in the parquet files.")


Loading results from: /Users/cyro/Documents/VSC/PopularityBias/data/wiki_full_bil/all_qa_8k
  ✓ neo | zero_shot | substring: 6,968 rows
  ✓ neo | bm25_plus top1 | substring: 6,968 rows
  ✓ neo | bm25_plus top3 | substring: 6,968 rows
  ✓ neo | ivfpq_low top1 | substring: 6,968 rows
  ✓ neo | ivfpq_low top3 | substring: 6,968 rows
  ✓ neo | ivfpq_high top1 | substring: 6,968 rows
  ✓ neo | ivfpq_high top3 | substring: 6,968 rows
  ✓ qwen | zero_shot | substring: 6,968 rows
  ✓ qwen | bm25_plus top1 | substring: 6,968 rows
  ✓ qwen | bm25_plus top3 | substring: 6,968 rows
  ✓ qwen | ivfpq_low top1 | substring: 6,968 rows
  ✓ qwen | ivfpq_low top3 | substring: 6,968 rows
  ✓ qwen | ivfpq_high top1 | substring: 6,968 rows
  ✓ qwen | ivfpq_high top3 | substring: 6,968 rows

✓ shared_setup complete — 14 runs loaded, 97,552 total rows
  decile_col resolved → 'decile_chunk_weighted'
Backends with retrieval metadata: []



## 1. Overall Recall@K per backend

In [2]:
if not RETRIEVAL_BACKENDS:
    print("No retrieval data — skipping §1.")
else:
    recall_rows = []
    for bk in RETRIEVAL_BACKENDS:
        sub = results_all_r[results_all_r['backend'] == bk]
        hits = sub['hit'].dropna()
        if hits.empty:
            continue
        n = len(hits)
        rate = hits.mean()
        se   = (rate * (1-rate) / n) ** 0.5
        recall_rows.append({'backend': bk, 'recall': rate, 'ci95': 1.96*se, 'n': n})
    rec_df = pd.DataFrame(recall_rows)

    fig, ax = plt.subplots(figsize=(max(5, len(rec_df)*1.4), 4))
    x = np.arange(len(rec_df))
    bars = ax.bar(x, rec_df['recall']*100,
                  yerr=rec_df['ci95']*100, capsize=5,
                  color=[backend_color(b) for b in rec_df['backend']], alpha=0.85)
    ax.set_xticks(x)
    ax.set_xticklabels(rec_df['backend'])
    ax.set_ylabel('Recall@K (%)')
    ax.set_ylim(0, 105)
    ax.yaxis.set_major_formatter(mticker.PercentFormatter())
    ax.set_title('Overall Retrieval Recall@K per Backend')
    for bar, row in zip(bars, rec_df.itertuples()):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                f"{row.recall:.1%}\n(n={row.n:,})", ha='center', va='bottom', fontsize=8)
    plt.tight_layout()
    plt.savefig(IMAGES_DIR / 'retrieval_recall_overall.png', bbox_inches='tight', dpi=150)
    plt.show()
    print(rec_df.to_string(index=False))


No retrieval data — skipping §1.


## 2. Per-decile retrieval recall — popularity bias in retrieval

In [3]:
if not RETRIEVAL_BACKENDS:
    print("No retrieval data — skipping §2.")
else:
    dcol = decile_col if decile_col in results_all_r.columns else 'decile'
    if dcol not in results_all_r.columns:
        print(f"Decile column '{dcol}' not found — skipping §2.")
    else:
        fig, ax = plt.subplots(figsize=(10, 5))

        decile_recall_tables = {}
        for bk in RETRIEVAL_BACKENDS:
            sub = results_all_r[results_all_r['backend'] == bk].copy()
            sub['_decile'] = sub[dcol].astype(float)
            rows = []
            for d in range(NUM_DECILES):
                bucket = sub[sub['_decile'].round() == d]
                hits = bucket['hit'].dropna()
                n = len(hits)
                if n == 0:
                    rows.append({'decile': d+1, 'recall': np.nan, 'ci95': np.nan, 'n': 0})
                    continue
                r = hits.mean()
                se = (r*(1-r)/n)**0.5
                rows.append({'decile': d+1, 'recall': r, 'ci95': 1.96*se, 'n': n})
            dt = pd.DataFrame(rows)
            decile_recall_tables[bk] = dt
            ax.errorbar(dt['decile'], dt['recall']*100, yerr=dt['ci95']*100,
                        marker='o', linewidth=2, markersize=6, capsize=4,
                        label=bk, color=backend_color(bk))

        ax.set_xticks(range(1, NUM_DECILES+1))
        ax.set_xlabel('Popularity decile (1=least popular → 10=most popular)')
        ax.set_ylabel('Recall@K (%)')
        ax.set_ylim(0, 105)
        ax.yaxis.set_major_formatter(mticker.PercentFormatter())
        ax.set_title('Per-Decile Retrieval Recall — Popularity Bias')
        ax.legend(title='backend', loc='best')
        plt.tight_layout()
        plt.savefig(IMAGES_DIR / 'retrieval_recall_by_decile.png', bbox_inches='tight', dpi=150)
        plt.show()

        for bk, dt in decile_recall_tables.items():
            print(f"\n--- {bk} ---")
            print(dt.set_index('decile').applymap(
                lambda v: f'{v:.1%}' if pd.notna(v) else '—'
            ).to_string())


No retrieval data — skipping §2.


## 3. Recall vs Accuracy scatter per decile

In [4]:
if not RETRIEVAL_BACKENDS:
    print("No retrieval data — skipping §3.")
else:
    dcol = decile_col if decile_col in results_all_r.columns else 'decile'
    if dcol not in results_all_r.columns:
        print(f"Decile column '{dcol}' not found — skipping §3.")
    else:
        for ev in EVALUATOR_KEYS:
            df_ev = results_all_r[results_all_r['evaluator'] == ev].copy()
            df_ev['_decile'] = df_ev[dcol].astype(float)

            ncols = min(len(RETRIEVAL_BACKENDS), 3)
            nrows = int(np.ceil(len(RETRIEVAL_BACKENDS) / ncols))
            fig, axes = plt.subplots(nrows, ncols,
                                     figsize=(5*ncols, 4.5*nrows), squeeze=False)

            for idx, bk in enumerate(RETRIEVAL_BACKENDS):
                r, c = divmod(idx, ncols)
                ax = axes[r][c]
                sub = df_ev[df_ev['backend'] == bk].copy()
                rows = []
                for d in range(NUM_DECILES):
                    bucket = sub[sub['_decile'].round() == d]
                    n = len(bucket)
                    if n == 0:
                        continue
                    acc  = bucket['evaluation_score'].mean() * 100
                    rec  = bucket['hit'].dropna().mean() * 100
                    rows.append({'decile': d+1, 'accuracy': acc, 'recall': rec, 'n': n})
                if not rows:
                    ax.axis('off'); continue
                dt = pd.DataFrame(rows)

                sc = ax.scatter(dt['recall'], dt['accuracy'],
                                c=dt['decile'], cmap='RdYlGn',
                                s=dt['n']/dt['n'].max()*300 + 20,
                                alpha=0.85, edgecolors='white', linewidth=0.5)
                plt.colorbar(sc, ax=ax, label='decile')
                # Annotate each point with its decile
                for _, row in dt.iterrows():
                    ax.annotate(int(row['decile']), (row['recall'], row['accuracy']),
                                textcoords='offset points', xytext=(4, 4), fontsize=7)
                ax.set_xlabel('Recall@K (%)')
                ax.set_ylabel('Accuracy (%)')
                ax.set_title(bk)
                ax.set_xlim(0, 105)
                ax.set_ylim(0, 105)

            for idx in range(len(RETRIEVAL_BACKENDS), nrows*ncols):
                r2, c2 = divmod(idx, ncols)
                axes[r2][c2].axis('off')

            fig.suptitle(f'Recall vs Accuracy per Decile — evaluator: {ev}', fontsize=12)
            plt.tight_layout()
            plt.savefig(IMAGES_DIR / f'recall_vs_accuracy_{ev}.png', bbox_inches='tight', dpi=150)
            plt.show()


No retrieval data — skipping §3.


## 4. Retrieval failure → output failure correlation

For each popularity decile, compare error rate when retrieval **missed** vs **hit** the target document. A large gap means retrieval quality directly drives generation quality.

In [5]:
if not RETRIEVAL_BACKENDS:
    print("No retrieval data — skipping §4.")
else:
    dcol = decile_col if decile_col in results_all_r.columns else 'decile'
    if dcol not in results_all_r.columns:
        print(f"Decile column not found — skipping §4.")
    else:
        for ev in EVALUATOR_KEYS:
            df_ev = results_all_r[results_all_r['evaluator'] == ev].copy()
            df_ev['_decile'] = df_ev[dcol].astype(float)
            df_ev['correct'] = df_ev['evaluation_score'].astype(bool)

            ncols = min(len(RETRIEVAL_BACKENDS), 3)
            nrows = int(np.ceil(len(RETRIEVAL_BACKENDS) / ncols))
            fig, axes = plt.subplots(nrows, ncols,
                                     figsize=(5.5*ncols, 4.5*nrows),
                                     sharey=True, squeeze=False)

            for idx, bk in enumerate(RETRIEVAL_BACKENDS):
                r, c = divmod(idx, ncols)
                ax = axes[r][c]
                sub = df_ev[df_ev['backend'] == bk].copy()
                sub = sub.dropna(subset=['hit'])

                rows = []
                for d in range(NUM_DECILES):
                    bucket = sub[sub['_decile'].round() == d]
                    miss = bucket[~bucket['hit'].astype(bool)]
                    hit  = bucket[bucket['hit'].astype(bool)]
                    err_miss = (~miss['correct']).mean()*100 if len(miss) else np.nan
                    err_hit  = (~hit['correct']).mean()*100  if len(hit)  else np.nan
                    rows.append({'decile': d+1,
                                 'err_rate_miss': err_miss,
                                 'err_rate_hit':  err_hit,
                                 'n_miss': len(miss), 'n_hit': len(hit)})
                dt = pd.DataFrame(rows)

                x = dt['decile'].values
                w = 0.38
                ax.bar(x - w/2, dt['err_rate_miss'], width=w,
                       label='Retrieval miss', color='#c0392b', alpha=0.85)
                ax.bar(x + w/2, dt['err_rate_hit'],  width=w,
                       label='Retrieval hit',  color='#27ae60', alpha=0.85)
                ax.set_title(bk)
                ax.set_xticks(x)
                ax.set_xlabel('Popularity decile')
                ax.set_ylim(0, 100)
                ax.yaxis.set_major_formatter(mticker.PercentFormatter())

            for idx in range(len(RETRIEVAL_BACKENDS), nrows*ncols):
                r2, c2 = divmod(idx, ncols)
                axes[r2][c2].axis('off')

            axes[0][0].set_ylabel('Error rate (% wrong answers)')
            handles, labels = axes[0][0].get_legend_handles_labels()
            fig.legend(handles, labels, loc='upper right')
            fig.suptitle(
                f'Error Rate: Retrieval Miss vs Hit per Decile — evaluator: {ev}\n'
                '(large gap = retrieval quality drives generation quality)',
                fontsize=12
            )
            plt.tight_layout()
            plt.savefig(IMAGES_DIR / f'retrieval_failure_correlation_{ev}.png', bbox_inches='tight', dpi=150)
            plt.show()


No retrieval data — skipping §4.


## 5. Popularity skew in retrieval errors

Among **wrong** answers: what fraction of the time was the top retrieved document *more* popular than the target document? A high fraction (>50%) means the retriever preferentially returns popular docs, crowding out the correct (less-popular) ones.

In [6]:
if not RETRIEVAL_BACKENDS:
    print("No retrieval data — skipping §5.")
else:
    dcol = decile_col if decile_col in results_all_r.columns else 'decile'
    has_pop = 'retrieved_doc_popularity' in results_all_r.columns and 'popularity_avg' in results_all_r.columns

    if not has_pop:
        print("Columns 'retrieved_doc_popularity' or 'popularity_avg' not found — skipping §5.")
    elif dcol not in results_all_r.columns:
        print(f"Decile column not found — skipping §5.")
    else:
        for ev in EVALUATOR_KEYS:
            df_ev = results_all_r[results_all_r['evaluator'] == ev].copy()
            df_ev['_decile'] = df_ev[dcol].astype(float)
            df_ev['correct'] = df_ev['evaluation_score'].astype(bool)

            fig, ax = plt.subplots(figsize=(10, 5))

            for bk in RETRIEVAL_BACKENDS:
                sub = df_ev[(df_ev['backend'] == bk) & (~df_ev['correct'])].copy()
                sub = sub.dropna(subset=['popularity_avg'])
                sub = sub[sub['retrieved_doc_popularity'].apply(
                    lambda x: isinstance(x, list) and len(x) > 0)]
                if sub.empty:
                    continue

                sub['max_ret_pop'] = sub['retrieved_doc_popularity'].apply(max)
                sub['ret_more_popular'] = sub['max_ret_pop'] > sub['popularity_avg']

                rows = []
                for d in range(NUM_DECILES):
                    bucket = sub[sub['_decile'].round() == d]
                    n = len(bucket)
                    frac = bucket['ret_more_popular'].mean() * 100 if n > 0 else np.nan
                    rows.append({'decile': d+1, 'frac': frac, 'n': n})
                dt = pd.DataFrame(rows)

                ax.plot(dt['decile'], dt['frac'], marker='o', linewidth=2, markersize=8,
                        label=bk, color=backend_color(bk))
                for _, row in dt.iterrows():
                    if not np.isnan(row['frac']):
                        ax.annotate(f"n={row['n']:.0f}",
                                    (row['decile'], row['frac']),
                                    textcoords='offset points', xytext=(0, 10),
                                    fontsize=7, ha='center', color='#666')

            ax.axhline(50, color='grey', linestyle='--', linewidth=0.8, alpha=0.5, label='50% baseline')
            ax.set_xticks(range(1, NUM_DECILES+1))
            ax.set_xlabel('Popularity decile (1=least popular → 10=most popular)')
            ax.set_ylabel('% wrong answers where retrieved doc is more popular')
            ax.set_ylim(0, 105)
            ax.yaxis.set_major_formatter(mticker.PercentFormatter())
            ax.set_title(f'Popularity Skew in Retrieval Errors — evaluator: {ev}')
            ax.legend()
            plt.tight_layout()
            plt.savefig(IMAGES_DIR / f'popularity_skew_retrieval_errors_{ev}.png', bbox_inches='tight', dpi=150)
            plt.show()


No retrieval data — skipping §5.


## 6. Error composition — retrieval miss vs hit among wrong answers

In [7]:
if not RETRIEVAL_BACKENDS:
    print("No retrieval data — skipping §6.")
else:
    for ev in EVALUATOR_KEYS:
        df_ev = results_all_r[results_all_r['evaluator'] == ev].copy()
        df_ev['correct'] = df_ev['evaluation_score'].astype(bool)

        datasets = (sorted(df_ev['dataset'].dropna().unique())
                    if 'dataset' in df_ev.columns else ['All'])

        for scope_name, scope_mask in [('All', pd.Series(True, index=df_ev.index))] +                                        [(ds, df_ev.get('dataset', pd.Series()) == ds)
                                        for ds in datasets if datasets != ['All']]:

            err_rows = []
            for bk in RETRIEVAL_BACKENDS:
                sub = df_ev[df_ev['backend'] == bk & scope_mask if scope_name != 'All'
                            else df_ev['backend'] == bk]
                if sub.empty:
                    continue
                errors = sub[~sub['correct']]
                n_err = len(errors)
                if n_err == 0:
                    err_rows.append({'backend': bk, 'pct_miss': 0, 'pct_hit': 0, 'n': 0})
                    continue
                valid = errors.dropna(subset=['hit'])
                miss_pct = (~valid['hit'].astype(bool)).mean() * 100 if len(valid) else np.nan
                hit_pct  =  valid['hit'].astype(bool).mean()  * 100 if len(valid) else np.nan
                err_rows.append({'backend': bk, 'pct_miss': miss_pct, 'pct_hit': hit_pct, 'n': n_err})

            if not err_rows:
                continue
            err_df = pd.DataFrame(err_rows).set_index('backend')

            fig, ax = plt.subplots(figsize=(max(5, len(err_df)*1.5), 4.5))
            x = np.arange(len(err_df))
            ax.bar(x, err_df['pct_miss'], label='Error + retrieval miss', color='#e74c3c', alpha=0.9)
            ax.bar(x, err_df['pct_hit'],  bottom=err_df['pct_miss'],
                   label='Error + retrieval hit',  color='#f39c12', alpha=0.9)

            for i, (bk, row) in enumerate(err_df.iterrows()):
                if not np.isnan(row['pct_miss']):
                    ax.text(i, row['pct_miss']/2,   f"{row['pct_miss']:.1f}%",
                            ha='center', va='center', fontsize=9, color='white')
                if not np.isnan(row['pct_hit']):
                    ax.text(i, row['pct_miss']+row['pct_hit']/2, f"{row['pct_hit']:.1f}%",
                            ha='center', va='center', fontsize=9, color='black')

            ax.set_xticks(x)
            ax.set_xticklabels(err_df.index)
            ax.set_ylabel('Share of wrong answers (%)')
            ax.set_ylim(0, 100)
            ax.yaxis.set_major_formatter(mticker.PercentFormatter())
            ax.set_title(f'Error Composition — {scope_name} | evaluator: {ev}')
            ax.legend(loc='upper right')
            plt.tight_layout()
            safe = scope_name.replace('/', '_')
            plt.savefig(IMAGES_DIR / f'error_composition_{safe}_{ev}.png', bbox_inches='tight', dpi=150)
            plt.show()


No retrieval data — skipping §6.


## 7. Three popularity-preference conditions across deciles

For each query, we compute the fraction of its retrieved documents that are more popular than the target article, then average across queries within a condition:

- **RED** `P(pop↑ | hit=0)` — wrongly-retrieved queries  
- **ORANGE** `P(pop↑ | correct=0)` — wrongly-answered queries  
- **PURPLE** `P(pop↑ | correct=1)` — correctly-answered queries  
- **GREEN** `P(pop↑ | hit=1)` — correctly-retrieved queries

A random retriever on a popularity-skewed corpus would show a descending curve from ~95% (decile 1, target is obscure → retrieved docs likely more popular) to ~5% (decile 10, target is already the most popular).

In [8]:
if not RETRIEVAL_BACKENDS or 'retrieved_doc_popularity' not in results_all_r.columns:
    print("Need retrieval metadata — skipping §7.")
else:
    dcol = decile_col if decile_col in results_all_r.columns else 'decile'
    if dcol not in results_all_r.columns:
        print(f"Decile column not found — skipping §7.")
    else:
        # ── helpers ──────────────────────────────────────────────────────────
        def _query_proportions(subset: pd.DataFrame) -> list:
            """Per-query fraction of retrieved docs more popular than target."""
            props = []
            for _, r in subset.iterrows():
                target_pop = r.get('popularity_avg')
                if pd.isna(target_pop):
                    continue
                pops = r.get('retrieved_doc_popularity', [])
                if not isinstance(pops, list) or len(pops) == 0:
                    continue
                props.append(float(np.mean([1.0 if p > float(target_pop) else 0.0 for p in pops])))
            return props

        def _prob_ci(props):
            n = len(props)
            if n == 0:
                return np.nan, np.nan, 0
            p = float(np.mean(props))
            ci = 1.96 * (p*(1-p)/n)**0.5
            return p*100, ci*100, n

        _random_base   = np.linspace(95, 5,   NUM_DECILES)
        _random_top    = np.linspace(100, 10,  NUM_DECILES)
        _random_bottom = np.linspace(90, 0,    NUM_DECILES)

        datasets = (sorted(results_all_r['dataset'].dropna().unique())
                    if 'dataset' in results_all_r.columns else ['All'])[:6]

        for ev in EVALUATOR_KEYS:
            df_ev = results_all_r[results_all_r['evaluator'] == ev].copy()
            df_ev['correct'] = df_ev['evaluation_score'].astype(bool)
            df_ev['_decile'] = df_ev[dcol].astype(float)

            for bk in RETRIEVAL_BACKENDS:
                sub_bk = df_ev[(df_ev['backend'] == bk) &
                               df_ev['retrieved_doc_popularity'].apply(
                                   lambda x: isinstance(x, list) and len(x) > 0)].copy()
                if sub_bk.empty:
                    continue

                # Build rows per dataset × decile
                rows = []
                for ds in datasets:
                    ds_df = sub_bk if ds == 'All' else sub_bk[sub_bk.get('dataset', pd.Series()) == ds]
                    if ds_df.empty:
                        continue
                    for d in range(NUM_DECILES):
                        bucket = ds_df[ds_df['_decile'].round() == d].copy()
                        if 'hit' in bucket.columns:
                            hit_s   = bucket['hit'].fillna(False).astype(bool)
                        else:
                            hit_s = pd.Series(False, index=bucket.index)
                        corr_s  = bucket['correct'].fillna(False).astype(bool)

                        p1, ci1, n1 = _prob_ci(_query_proportions(bucket[~hit_s]))
                        p2, ci2, n2 = _prob_ci(_query_proportions(bucket[~corr_s]))
                        p3, ci3, n3 = _prob_ci(_query_proportions(bucket[corr_s]))
                        p4, ci4, n4 = _prob_ci(_query_proportions(bucket[hit_s]))

                        rows.append({'dataset': ds, 'decile': d+1,
                                     'p1':p1,'ci1':ci1,'n1':n1,
                                     'p2':p2,'ci2':ci2,'n2':n2,
                                     'p3':p3,'ci3':ci3,'n3':n3,
                                     'p4':p4,'ci4':ci4,'n4':n4})

                if not rows:
                    continue
                cond_df = pd.DataFrame(rows)

                ncols = min(3, len(datasets))
                nrows = int(np.ceil(len(datasets) / ncols))
                fig, axes = plt.subplots(nrows, ncols,
                                         figsize=(7.5*ncols, 5.5*nrows),
                                         sharey=True, squeeze=False)

                for idx, ds in enumerate(datasets):
                    r, c = divmod(idx, ncols)
                    ax = axes[r][c]
                    cur = cond_df[cond_df['dataset'] == ds].sort_values('decile')
                    if cur.empty:
                        ax.axis('off'); continue

                    x = cur['decile'].values
                    ax.fill_between(range(1, NUM_DECILES+1), _random_top, _random_bottom,
                                    color='black', alpha=0.08, label='random field', zorder=1)

                    _cond_cfg = [
                        ('p1','ci1','#e74c3c','-','o',  r'$P(\mathrm{pop}_{\uparrow}|\mathrm{hit}=0)$'),
                        ('p2','ci2','#f39c12','--','s', r'$P(\mathrm{pop}_{\uparrow}|\mathrm{correct}=0)$'),
                        ('p3','ci3','#8e44ad','-.','D', r'$P(\mathrm{pop}_{\uparrow}|\mathrm{correct}=1)$'),
                        ('p4','ci4','#27ae60','--','^', r'$P(\mathrm{pop}_{\uparrow}|\mathrm{hit}=1)$'),
                    ]
                    for pc, cic, col, ls, mk, lbl in _cond_cfg:
                        ax.errorbar(x, cur[pc].values, yerr=cur[cic].values,
                                    color=col, linestyle=ls, marker=mk,
                                    linewidth=2.2, markersize=5.5, capsize=2.8, label=lbl, zorder=5)

                    ax.plot(range(1, NUM_DECILES+1), _random_base, ':', color='black',
                            linewidth=1.5, alpha=0.8, label='random baseline', zorder=6)
                    ax.set_title(ds)
                    ax.set_xticks(range(1, NUM_DECILES+1))
                    ax.set_ylim(0, 100)
                    ax.yaxis.set_major_formatter(mticker.PercentFormatter())
                    ax.grid(True, axis='y', alpha=0.25)

                for idx in range(len(datasets), nrows*ncols):
                    r2, c2 = divmod(idx, ncols)
                    axes[r2][c2].axis('off')

                axes[0][0].set_ylabel('Probability (%)')
                if nrows > 1:
                    axes[-1][0].set_ylabel('Probability (%)')
                for c in range(ncols):
                    axes[-1][c].set_xlabel('Popularity decile')

                handles, labels = axes[0][0].get_legend_handles_labels()
                uniq = {}
                for h, l in zip(handles, labels):
                    if l not in uniq: uniq[l] = h
                fig.legend(list(uniq.values()), list(uniq.keys()),
                           loc='upper right', fontsize=9)

                fig.suptitle(
                    f'{bk}: Three Popularity-Preference Conditions — evaluator: {ev}',
                    fontsize=14, fontweight='bold', y=1.01
                )
                plt.tight_layout()
                fname = f'pop_pref_three_cond_{bk}_{ev}.png'
                plt.savefig(IMAGES_DIR / fname, bbox_inches='tight', dpi=150)
                plt.show()
                print(f"Saved -> {fname}")


Need retrieval metadata — skipping §7.
